# ESG Media Sentiment Analysis (GDELT GKG)

**Raw dataset:** one 15-minute GDELT GKG batch (`20261117200000.gkg.csv`).

This notebook loads the raw GKG file, assigns the standard GDELT 2.0 GKG column headers, and displays the data.

## User Parameters

Edit the values in the cell below to control which data the pipeline processes. Then run all cells.

- **date_range**: inclusive `(start, end)` dates, ISO format (`"2026-11-17"`) or GDELT timestamp (`"20261117200000"`). Use `None` to leave a bound open.
- **continents**: e.g. `["North America", "Europe", "Asia"]`, or `["ALL"]` for no filter.
- **countries**: e.g. `["US", "GB", "ID"]` (ISO codes) or country names, or `["ALL"]` for no filter.
- **aggregation_freq**: `"1D"` (daily), `"1W"` (weekly), `"1M"` (monthly). Default `"1W"`.
- **min_articles**: minimum articles required for a (country, time) bucket to be colored on the map. Buckets below this are hidden as too noisy. Set `0` to disable.

In [1]:
# ============================================================
# USER PARAMETERS - edit these to control the pipeline
# ============================================================

# Inclusive date range (start, end). None = no bound.
# ISO format "2026-11-17" or GDELT timestamp "20261117200000".
date_range = ("20251117200000", "20251217200000")

# Target continents, e.g. ["North America", "Europe", "Asia"].
# Use ["ALL"] for no continent filter.
continents = ["Europe"]

# Target countries (ISO codes or names), e.g. ["US", "GB", "ID"].
# Use ["ALL"] for no country filter.
countries = ["ALL"]

# Aggregation frequency: "1D" daily, "1W" weekly, "1M" monthly.
aggregation_freq = "1W"

# Minimum articles required for a map bucket to be colored (0 = disable mask).
min_articles = 3

# Apply the user parameters to the config module
import config
config.date_range = date_range
config.continents = continents
config.countries = countries
config.aggregation_freq = aggregation_freq
config.MIN_ARTICLES = min_articles

print('date_range:', config.date_range)
print('continents:', config.continents)
print('countries:', config.countries)
print('aggregation_freq:', config.aggregation_freq)
print('min_articles:', config.MIN_ARTICLES)

date_range: ('20251117200000', '20251217200000')
continents: ['Europe']
countries: ['ALL']
aggregation_freq: 1D


In [2]:
# Download raw GDELT GKG batches based on the date_range above
# ============================================================
# DOWNLOAD SETTINGS - edit these to control downloading
# ============================================================
# Set to True to download data, False to skip (use existing files).
download_enabled = False

# Download mode:
#   'daily'  - one batch per day (23:45) in the date range
#   'all'    - every 15-minute batch in the date range (96/day)
#   'latest' - the most recent available batch (ignores date_range)
download_mode = 'daily'

# Set to True to re-download and overwrite files that already exist.
# False (default) keeps existing files and only downloads missing ones.
force_download = False

from download_raw import download_batch, download_date_range, download_all_batches, download_latest_batch

if download_enabled:
    if download_mode == 'daily':
        start, end = config.date_range
        if start is None or end is None:
            raise ValueError('download_mode=daily requires both date_range bounds (set them in the parameters cell)')
        paths = download_date_range(start, end, force=force_download)
    elif download_mode == 'all':
        start, end = config.date_range
        if start is None or end is None:
            raise ValueError('download_mode=all requires both date_range bounds (set them in the parameters cell)')
        paths = download_all_batches(start, end, force=force_download)
    elif download_mode == 'latest':
        paths = [download_latest_batch(force=force_download)]
    else:
        raise ValueError(f"Unknown download_mode: {download_mode!r} (use 'daily', 'all', or 'latest')")
    print('downloaded:', [p.name for p in paths])
else:
    print('download disabled - using existing raw files')

print('raw data dir:', config.raw_data_dir)
print('existing raw files:', [f.name for f in config.raw_data_dir.glob('*.gkg.csv')])


download disabled - using existing raw files
raw data dir: /home/theodoret/Downloads/Interdiscplinarly/main/data/raw
existing raw files: ['20251126234500.gkg.csv', '20251121234500.gkg.csv', '20251210234500.gkg.csv', '20251205234500.gkg.csv', '20251212234500.gkg.csv', '20251213234500.gkg.csv', '20251216234500.gkg.csv', '20251215234500.gkg.csv', '20251211234500.gkg.csv', '20251217234500.gkg.csv', '20251118234500.gkg.csv', '20251130234500.gkg.csv', '20251124234500.gkg.csv', '20251119234500.gkg.csv', '20261117200000.gkg.csv', '20251206234500.gkg.csv', '20251204234500.gkg.csv', '20251203234500.gkg.csv', '20251128234500.gkg.csv', '20251127234500.gkg.csv', '20251208234500.gkg.csv', '20251129234500.gkg.csv', '20251122234500.gkg.csv', '20251207234500.gkg.csv', '20251120234500.gkg.csv', '20251214234500.gkg.csv', '20251125234500.gkg.csv', '20251209234500.gkg.csv', '20251123234500.gkg.csv', '20251117234500.gkg.csv', '20251201234500.gkg.csv', '20251202234500.gkg.csv']


In [3]:
import sys
from pathlib import Path

# Make config.py importable
sys.path.insert(0, str(Path.cwd()))

import config
import pandas as pd

print('raw dir:', config.raw_data_dir)

raw dir: /home/theodoret/Downloads/Interdiscplinarly/main/data/raw


In [4]:
# Locate and load the raw GKG files via the fetch_raw module
from fetch_raw import fetch_raw_gkg, find_raw_gkg_files

df, raw_paths = fetch_raw_gkg(date_range=config.date_range)
print('raw GKG files found:', [f.name for f in find_raw_gkg_files()])
print('loaded files:', len(raw_paths), '->', [p.name for p in raw_paths])
print('shape:', df.shape)
print('columns:', list(df.columns))


raw GKG files found: ['20251117234500.gkg.csv', '20251118234500.gkg.csv', '20251119234500.gkg.csv', '20251120234500.gkg.csv', '20251121234500.gkg.csv', '20251122234500.gkg.csv', '20251123234500.gkg.csv', '20251124234500.gkg.csv', '20251125234500.gkg.csv', '20251126234500.gkg.csv', '20251127234500.gkg.csv', '20251128234500.gkg.csv', '20251129234500.gkg.csv', '20251130234500.gkg.csv', '20251201234500.gkg.csv', '20251202234500.gkg.csv', '20251203234500.gkg.csv', '20251204234500.gkg.csv', '20251205234500.gkg.csv', '20251206234500.gkg.csv', '20251207234500.gkg.csv', '20251208234500.gkg.csv', '20251209234500.gkg.csv', '20251210234500.gkg.csv', '20251211234500.gkg.csv', '20251212234500.gkg.csv', '20251213234500.gkg.csv', '20251214234500.gkg.csv', '20251215234500.gkg.csv', '20251216234500.gkg.csv', '20251217234500.gkg.csv', '20261117200000.gkg.csv']
loaded files: 31 -> ['20251117234500.gkg.csv', '20251118234500.gkg.csv', '20251119234500.gkg.csv', '20251120234500.gkg.csv', '20251121234500.gkg.c

In [5]:
# Preprocess: NA handling, dedup, and persistence via the preprocess module
from preprocess import preprocess, save_processed

df_clean = preprocess(df)
print('clean shape:', df_clean.shape)

clean_path = save_processed(df_clean)
print('saved to:', clean_path)

preprocess: dropped 0 rows with missing DATE
preprocess: 26791 rows without V2Tone (kept for volume stats)
preprocess: 0 rows flagged missing_identifier
preprocess: dropped 0 duplicate rows on ['DocumentIdentifier', 'DATE']
clean shape: (39066, 28)
preprocess: saved 39066 rows x 8 cols -> /home/theodoret/Downloads/Interdiscplinarly/main/data/processed/gkg_clean.csv
saved to: /home/theodoret/Downloads/Interdiscplinarly/main/data/processed/gkg_clean.csv


In [6]:
# ESG classification: multi-label tagging via the esg_classify module
from esg_classify import classify

df_esg = classify(df_clean)
print('esg shape:', df_esg.shape)

pillar_cols = [f'is_{p}' for p in config.esg_categories]
print('tagged counts:')
print(df_esg[pillar_cols].sum())
print('multi-label rows:', (df_esg[pillar_cols].sum(axis=1) > 1).sum())
print('unclassified rows:', (df_esg[pillar_cols].sum(axis=1) == 0).sum())

esg shape: (39066, 32)
tagged counts:
is_Environmental    13911
is_Social           26889
is_Governance       32814
dtype: int64
multi-label rows: 27607
unclassified rows: 5339


In [7]:
# Tone / sentiment extraction via the tone module
from tone import extract_tone

df_tone = extract_tone(df_esg)
print('tone shape:', df_tone.shape)

tone_cols = ['tone', 'positive', 'negative', 'polarity']
print('rows with tone:', df_tone['tone'].notna().sum())
print(df_tone[tone_cols].describe())

tone shape: (39066, 36)
rows with tone: 12275
               tone      positive      negative      polarity
count  12275.000000  12275.000000  12275.000000  12275.000000
mean       2.567088      5.097108      7.761059     -2.663951
std        1.464075      5.296045      9.752285      7.547918
min        1.000000      0.000000      0.000000    -30.000000
25%        1.000000      0.000000      0.000000     -6.000000
50%        3.000000      3.000000      1.000000      0.000000
75%        4.000000     11.000000     15.000000      0.000000
max        4.000000     12.000000     31.000000     12.000000


In [8]:
# Media source geographic attribution via the geo_attr module
from geo_attr import attribute, filter_by_countries

df_geo = attribute(df_tone)
print('geo shape (before filter):', df_geo.shape)

# Apply user country / continent filters
df_geo = filter_by_countries(df_geo, countries=config.countries, continents=config.continents)
print('geo shape (after filter):', df_geo.shape)

print('source_country coverage:', df_geo['source_country'].notna().sum(), '/', len(df_geo))
print('location_country coverage:', df_geo['location_country'].notna().sum(), '/', len(df_geo))
print('attributed_country coverage:', (df_geo['attributed_country'] != config.UNKNOWN_SOURCE).sum(), '/', len(df_geo))
print('\ntop attributed countries:')
print(df_geo['attributed_country'].value_counts().head(10))

geo shape (before filter): (39066, 40)
geo shape (after filter): (5211, 40)
source_country coverage: 2972 / 5211
location_country coverage: 4479 / 5211
attributed_country coverage: 5211 / 5211

top attributed countries:
attributed_country
United Kingdom    3223
Ireland            327
France             254
Germany            236
Ukraine            187
Russia             161
Italy              151
Spain               78
Austria             70
Greece              60
Name: count, dtype: int64


In [9]:
# Metric aggregation: ESG x country x time window via the aggregate module
from aggregate import aggregate, save_aggregated

df_agg = aggregate(df_geo, freq=config.aggregation_freq)
print('agg shape:', df_agg.shape)
print('freq:', config.aggregation_freq)
print('categories:', sorted(df_agg['esg_category'].unique()))
print('countries:', df_agg['country'].nunique())
print('date range:', df_agg['date'].min(), '->', df_agg['date'].max())
print('\nhead:')
print(df_agg.head(10).to_string(index=False))

agg_path = save_aggregated(df_agg)
print('saved to:', agg_path)

agg shape: (1757, 8)
freq: 1D
categories: ['Composite', 'Environmental', 'Governance', 'Social']
countries: 42
date range: 2025-11-17 00:00:00 -> 2025-12-17 00:00:00

head:
      date country  esg_category  tone_mean  tone_median  tone_std  n_with_tone  n_articles
2025-11-17 Austria     Composite        4.0          4.0       NaN            1           1
2025-11-17 Austria Environmental        4.0          4.0       NaN            1           1
2025-11-17 Austria    Governance        4.0          4.0       NaN            1           1
2025-11-17 Austria        Social        4.0          4.0       NaN            1           1
2025-11-17 Belarus     Composite        4.0          4.0       NaN            1           1
2025-11-17 Belarus    Governance        4.0          4.0       NaN            1           1
2025-11-17 Belarus        Social        4.0          4.0       NaN            1           1
2025-11-17 Denmark     Composite        NaN          NaN       NaN            0           3

In [10]:
# Interactive world map visualization via the viz_map module
from viz_map import build_choropleth, build_date_slider_figure, build_category_selector_figure, build_time_series_figure, save_figure

# Latest-date choropleth for each ESG category
for cat in config.esg_categories + ['Composite']:
    fig = build_choropleth(df_agg, category=cat)
    save_figure(fig, f'map_{cat.lower()}')

# Date-slider figure (Composite)
slider_fig = build_date_slider_figure(df_agg, category='Composite')
save_figure(slider_fig, 'map_composite_slider')

# Category-switcher figure (dropdown: E / S / G / Composite)
selector_fig = build_category_selector_figure(df_agg)
save_figure(selector_fig, 'map_category_selector')

# Time-series figure: category dropdown + date slider (change over time)
time_fig = build_time_series_figure(df_agg)
save_figure(time_fig, 'map_time_series')

# Show the time-series map inline
build_time_series_figure(df_agg)

viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_environmental.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_social.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_governance.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_composite.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_composite_slider.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_category_selector.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_time_series.html


In [11]:
# Show the data
df_esg.head()

,GKGRecordID,DATE,NumArticles,SourceCommonName,DocumentIdentifier,V1Counts,V2Counts,V1Themes,V2Themes,V1Locations,...,V2SocialImage,V1Quotations,V2Quotations,V1AllNames,V2AllNames,missing_identifier,is_Environmental,is_Social,is_Governance,esg_pillars
0,20251117234500-0,20251117234500,1,batonrougepost.com,http://www.batonrougepost.com/news/278700465/f...,,,,,"3#Washington, Washington, United States#US#USD...",...,,"Brett Davis-Imagn Images,44;Atlanta Falcons,93...","13,of 16 passes for,630;16,passes for 175 yard...",,<PAGE_TITLE>Falcons QB Michael Penix Jr. (knee...,False,False,False,False,
1,20251117234500-1,20251117234500,1,wsoctv.com,https://www.wsoctv.com/news/local/9-investigat...,,,TRIAL;TAX_FNCACT;TAX_FNCACT_ATTORNEY;LEGISLATI...,"EPU_ECONOMY_HISTORIC,912;WB_2745_JOB_QUALITY_A...","2#North Carolina, United States#US#USNC#35.641...",...,,"Jason Stoogenke,48;Michael Easley,467;Eastern ...","9,attorney Jason Stoogenke says,23;700,dollars...",,<PAGE_PRECISEPUBTIMESTAMP>20251117230700</PAGE...,False,False,True,True,Governance;Social
2,20251117234500-2,20251117234500,1,kltv.com,https://www.kltv.com/2025/11/17/roads-closed-a...,,,MARITIME_INCIDENT;MARITIME;MANMADE_DISASTER_IM...,"WB_135_TRANSPORT,255;WB_1803_TRANSPORT_INFRAST...","4#Athens, AttikíR, Greece#GR#GR35#37.9833#23.7...",...,,"Athens Public Safety,136","175,West at the railroad,169;",,<PAGE_ALTURL_AMP>https://www.kltv.com/2025/11/...,False,True,True,False,Environmental;Social
3,20251117234500-3,20251117234500,1,98kupd.com,https://98kupd.com/vinnie-paul-recipes-to-be-s...,,,TAX_FNCACT;TAX_FNCACT_DRUMMER;TAX_FNCACT_MONK;...,"TAX_FNCACT_MONK,79;TAX_ETHNICITY_BLACK,824;WMD...","2#Texas, United States#US#USTX#31.106#-97.6475#TX",...,,"Vinnie Paul,61;Metal Hammer,120;Team Rock,163;...",,,<PAGE_PRECISEPUBTIMESTAMP>20251117200000</PAGE...,False,True,True,True,Environmental;Governance;Social
4,20251117234500-4,20251117234500,1,iheart.com,https://991kglx.iheart.com/content/2025-11-17-...,,,TAX_FNCACT;TAX_FNCACT_ARTISTS;EPU_ECONOMY_HIST...,"WB_678_DIGITAL_GOVERNMENT,1071;WB_694_BROADCAS...",,...,,"Luke Combs,70;Miranda Lambert,91;Scotty McCree...",,,<PAGE_LINKS>https://country.iheart.com/content...,False,False,False,True,Governance


In [12]:
# Focus on the core columns from the plan
core = ['DATE', 'DocumentIdentifier', 'V2Themes', 'V2Locations', 'V2Tone']
df[core].head(10)

,DATE,DocumentIdentifier,V2Themes,V2Locations,V2Tone
0,20251117234500,http://www.batonrougepost.com/news/278700465/f...,,"2#Indiana, United States#US#USIN##39.8647#-86....",
1,20251117234500,https://www.wsoctv.com/news/local/9-investigat...,"EPU_ECONOMY_HISTORIC,912;WB_2745_JOB_QUALITY_A...","2#North Carolina, United States#US#USNC##35.64...",
2,20251117234500,https://www.kltv.com/2025/11/17/roads-closed-a...,"WB_135_TRANSPORT,255;WB_1803_TRANSPORT_INFRAST...","4#Athens, AttikíR, Greece#GR#GR35#16716#37.983...",
3,20251117234500,https://98kupd.com/vinnie-paul-recipes-to-be-s...,"TAX_FNCACT_MONK,79;TAX_ETHNICITY_BLACK,824;WMD...","2#Texas, United States#US#USTX##31.106#-97.647...",3#11#1#2010#82;1#0#0#2004#717
4,20251117234500,https://991kglx.iheart.com/content/2025-11-17-...,"WB_678_DIGITAL_GOVERNMENT,1071;WB_694_BROADCAS...",,
5,20251117234500,https://www.kfvs12.com/2025/11/17/aaa-tips-ahe...,"TAX_DISEASE_COVID_19,1815;WB_165_AIR_TRANSPORT...",1#Americans#US#US##39.828175#-98.5795#US#193;1...,4#11#25#0#3119;4#12#1#0#3148
6,20251117234500,https://929thelake.com/ixp/478/p/brian-may-adm...,"TAX_FNCACT_GUITARIST,693;AFFECT,946;GENERAL_HE...",,
7,20251117234500,https://b104.iheart.com/featured/ryan-seacrest...,,,
8,20251117234500,https://timesofindia.indiatimes.com/city/manga...,"BAN,949;APPOINTMENT,655;TERROR,2230;WB_2467_TE...","4#Red Fort, Delhi, India#IN#IN07#17911#28.6333...",4#1#6#0#298
9,20251117234500,https://929thelake.com/louisiana-2025-infrastr...,"WB_137_WATER,166;WB_137_WATER,851;WB_137_WATER...","2#Louisiana, United States#US#USLA##31.1801#-9...",1#0#0#2012#1264
